# Single-line CAD text similarity lab

Finds characters drawn as raw vector primitives (line/curve/rect/quad) rather than native text or
raster images, by recognizing a character as a specific, repeatable *sequence* of consecutive
`Vector`s (by `seqno`, PDF content-stream draw order).

Workflow: load ground-truth vector-labelled clusters (a single JSON file produced directly by
`python scripts/label/vector_label.py PDF --page N`, default `outputs/labels/<pdf stem>.json`),
encode each one as a scale/rotation/
translation-invariant chain of deltas between consecutive flattened line segments (length-ratio,
angle change, gap angle, gap distance) plus its item-type/operation-count signature, save these as
character **templates** (JSON), then scan a page for candidate seqno-contiguous vector runs and
slide each template's window across them to detect character occurrences.

Notebook-only prototype (no new importable library module) -- fixed global tolerances, not learned
per template from multiple examples. See `rastervec/notebooks/vector_similarity_lab.ipynb` for the
sibling (unrelated) per-vector shape-similarity lab this borrows its cell conventions from.

## 0 - Config

In [ ]:
from pathlib import Path

PDF_PATH = None              # None -> first PDF under references/
LABEL_JSON_PATH = None       # scripts/label/vector_label.py's own output JSON;
                              # None -> outputs/labels/<PDF_PATH stem>.json (that tool's own default)
TEMPLATE_JSON_PATH = None    # None -> alongside LABEL_JSON_PATH, "<stem>_char_templates.json"

BEZIER_SUBDIVISIONS = 8      # fixed-count curve flattening, adequate at glyph scale

LENGTH_DIFF_TOLERANCE_PCT = 0.15
ANGLE_DIFF_TOLERANCE_DEG = 8.0
GAP_ANGLE_TOLERANCE_DEG = 8.0
GAP_DIST_TOLERANCE_PCT = 0.15

SPATIAL_CLUSTER_GAP = 5.0    # bbox-gap threshold (pdf points) for the candidate spatial pre-pass
MAX_SEQNO_RANK_GAP = 1       # how many non-member seqnos may be interleaved within a candidate run

DETECT_PAGE_INDICES = None   # None -> every page that has a vector label
MAX_MATCHES_SHOWN = 30
RENDER_DPI = 150

## 1 - Path bootstrap

In [ ]:
import os, sys
_root = os.path.abspath(os.path.join('../..'))
if _root not in sys.path:
    sys.path.append(_root)

## 2 - Resolve paths + load ground-truth labels

The ground-truth source is the single JSON file `scripts/label/vector_label.py` writes directly
(`python scripts/label/vector_label.py PDF --page N`, default `outputs/labels/<pdf stem>.json`) --
not a `master_label.py` folder. That file's own `pdf_path` (the PDF it was actually labelled
against) is authoritative and overrides `PDF_PATH` if the two disagree.

In [ ]:
from rastervec.Evaluation.Labelling.label_schema import load_labels, path_signature
from rastervec.P1_Reading_Native.reader import Reader
from rastervec.P1_Reading_Native.vector_extract import extract_vectors

if PDF_PATH is None:
    refs = sorted((Path(_root) / 'references').glob('*.pdf'))
    assert refs, 'no PDFs found under references/'
    PDF_PATH = str(refs[0])

if LABEL_JSON_PATH is None:
    LABEL_JSON_PATH = str(Path(_root) / 'outputs' / 'labels' / f'{Path(PDF_PATH).stem}.json')

labels = load_labels(LABEL_JSON_PATH)
if labels.pdf_path and Path(labels.pdf_path).resolve() != Path(PDF_PATH).resolve():
    print(f'note: {LABEL_JSON_PATH} was labelled against {labels.pdf_path!r} -- using that instead of PDF_PATH')
    PDF_PATH = labels.pdf_path

if TEMPLATE_JSON_PATH is None:
    TEMPLATE_JSON_PATH = str(Path(LABEL_JSON_PATH).with_name(Path(LABEL_JSON_PATH).stem + '_char_templates.json'))

print('PDF_PATH           =', PDF_PATH)
print('LABEL_JSON_PATH     =', LABEL_JSON_PATH)
print('TEMPLATE_JSON_PATH  =', TEMPLATE_JSON_PATH)

vector_entries = [e for e in labels.entries if e.source == 'vector']
print(f'{len(vector_entries)} vector-labelled entry(ies) in {LABEL_JSON_PATH}')

## 3 - Geometry: flatten items to segments

Every `Vector.items` entry becomes an ordered list of straight-line segments in absolute page
space: `"l"` is already one segment; `"re"`/`"qu"` become their 4 edges (same corner order
`label_schema.geometry_annotations_for_vector` uses, for consistency); `"c"` (cubic bezier) is
subdivided via De Casteljau into `BEZIER_SUBDIVISIONS` segments. A whole labelled/candidate
cluster (Vectors pre-sorted by `seqno`) flattens to one single segment sequence spanning every one
of its Vectors -- deltas are computed across the *whole cluster*, not per-Vector, matching how the
strokes of a character are drawn as several consecutive Vectors.

In [ ]:
def _bezier_point(p0, p1, p2, p3, t):
    mt = 1.0 - t
    x = mt**3 * p0[0] + 3 * mt**2 * t * p1[0] + 3 * mt * t**2 * p2[0] + t**3 * p3[0]
    y = mt**3 * p0[1] + 3 * mt**2 * t * p1[1] + 3 * mt * t**2 * p2[1] + t**3 * p3[1]
    return (x, y)


def _flatten_bezier(p0, p1, p2, p3, n=BEZIER_SUBDIVISIONS):
    pts = [_bezier_point(p0, p1, p2, p3, i / n) for i in range(n + 1)]
    return [(pts[i], pts[i + 1]) for i in range(n)]


def _polygon_edges(corners):
    return [(a, b) for a, b in zip(corners, corners[1:] + corners[:1])]


def vector_to_segments(v):
    """One Vector's items, flattened into an ordered list of ((x0,y0),(x1,y1)) segments."""
    segments = []
    for item in v.items:
        kind = item[0]
        if kind == 'l':
            segments.append((item[1], item[2]))
        elif kind == 're':
            x0, y0, x1, y1 = item[1]
            segments.extend(_polygon_edges([(x0, y0), (x1, y0), (x1, y1), (x0, y1)]))
        elif kind == 'qu':
            segments.extend(_polygon_edges(list(item[1])))
        elif kind == 'c':
            p0, p1, p2, p3 = item[1], item[2], item[3], item[4]
            segments.extend(_flatten_bezier(p0, p1, p2, p3))
    return segments


def cluster_to_segments(vectors):
    """Vectors, assumed already sorted by seqno -- one flat segment sequence spanning all of them."""
    segments = []
    for v in vectors:
        segments.extend(vector_to_segments(v))
    return segments

## 4 - Delta encoding

For each pair of consecutive segments in a cluster's flattened sequence: `% length difference`,
`angle difference` between the two segments' own directions, the `gap angle` from segment A's
endpoint to segment B's start point measured *relative to A's own direction* (so it's itself
rotation-invariant, not an absolute page angle), and the `gap distance` between A's endpoint and
B's start point as a percentage of A's own length. The resulting chain (`N-1` entries for `N`
segments) is scale/rotation/translation invariant by construction and has no single-segment
"baseline" entry to exclude -- every entry is already a pairwise delta.

In [ ]:
import math


def _seg_vector(seg):
    (x0, y0), (x1, y1) = seg
    return (x1 - x0, y1 - y0)


def _seg_length(seg):
    dx, dy = _seg_vector(seg)
    return math.hypot(dx, dy)


def _seg_angle_deg(seg):
    dx, dy = _seg_vector(seg)
    return math.degrees(math.atan2(dy, dx))


def _normalize_angle_deg(deg):
    """Wrap an angle (or angle difference) to (-180, 180]."""
    return (deg + 180.0) % 360.0 - 180.0


def segment_delta(a, b):
    """(length_diff_pct, angle_diff_deg, gap_angle_deg, gap_dist_pct) between consecutive segments a -> b."""
    len_a, len_b = _seg_length(a), _seg_length(b)
    length_diff_pct = (len_b - len_a) / len_a if len_a > 1e-9 else 0.0

    angle_a, angle_b = _seg_angle_deg(a), _seg_angle_deg(b)
    angle_diff_deg = _normalize_angle_deg(angle_b - angle_a)

    a_end, b_start = a[1], b[0]
    gap_dx, gap_dy = b_start[0] - a_end[0], b_start[1] - a_end[1]
    gap_dist = math.hypot(gap_dx, gap_dy)
    gap_dist_pct = gap_dist / len_a if len_a > 1e-9 else 0.0
    absolute_gap_angle = math.degrees(math.atan2(gap_dy, gap_dx)) if gap_dist > 1e-9 else angle_a
    gap_angle_deg = _normalize_angle_deg(absolute_gap_angle - angle_a)

    return (length_diff_pct, angle_diff_deg, gap_angle_deg, gap_dist_pct)


def delta_chain(segments):
    return [segment_delta(segments[i], segments[i + 1]) for i in range(len(segments) - 1)]

## 5 - Templates: build ground-truth character templates

`item_type_signature` (op-count/type Counter per Vector) is reused as-is from
`P3_Vector_Parsing/FastIntoPaddle/similarity.py` -- it's exactly the "Vector is_similar: check
operation count and types" check. A template stores it two ways: one combined total (the
cluster-level fast-reject gate) and one per-Vector, in seqno order (needed so a candidate window's
Vectors line up position-for-position with the template's own before comparing deltas).

In [ ]:
from collections import Counter

from rastervec.P3_Vector_Parsing.FastIntoPaddle.similarity import item_type_signature


def _sig_key(sig):
    """Normalize a signature (list/tuple of [kind, count] pairs) to a hashable tuple-of-tuples,
    regardless of whether it was just built in this session (tuples) or round-tripped through
    JSON (lists) -- `("l", 4) != ["l", 4]` would otherwise silently break every comparison below
    once templates are reloaded from disk."""
    return tuple((kind, count) for kind, count in sig)


def _combined_signature(vectors):
    counts = Counter()
    for v in vectors:
        counts.update(dict(item_type_signature(v)))
    return tuple(sorted(counts.items()))


def resolve_label_vectors(entry, sig_to_vector):
    """A LabelEntry's vector_signatures, resolved back to real Vectors and sorted by seqno
    (vector_signatures itself is sorted by signature string, not draw order)."""
    vectors = [sig_to_vector[s] for s in entry.vector_signatures if s in sig_to_vector]
    missing = len(entry.vector_signatures) - len(vectors)
    if missing:
        print(f"warning: label {entry.label_id!r} ({entry.text!r}) missing {missing} vector(s) "
              f"on page {entry.page_index} -- re-extraction may be stale")
    vectors.sort(key=lambda v: v.seqno)
    return vectors


def build_template(entry, resolved_vectors):
    segments = cluster_to_segments(resolved_vectors)
    total_sig = _combined_signature(resolved_vectors)
    return {
        'label_id': entry.label_id,
        'text': entry.text,
        'page_index': entry.page_index,
        'total_item_signature': list(total_sig),
        'per_vector_item_signatures': [list(item_type_signature(v)) for v in resolved_vectors],
        'n_vectors': len(resolved_vectors),
        'n_items': sum(count for _, count in total_sig),
        'n_segments': len(segments),
        'deltas': delta_chain(segments),
    }


def build_all_templates(labels, pdf_path):
    entries = [e for e in labels.entries if e.source == 'vector']
    page_indices = sorted({e.page_index for e in entries})
    sig_to_vector = {}
    with Reader(pdf_path) as reader:
        for page_index in page_indices:
            page = reader.get_page(page_index)
            for v in extract_vectors(page):
                sig_to_vector[path_signature(v)] = v

    templates = []
    source_vectors = {}
    for entry in entries:
        resolved = resolve_label_vectors(entry, sig_to_vector)
        if not resolved:
            print(f"skipping label {entry.label_id!r} ({entry.text!r}): no vectors resolved")
            continue
        templates.append(build_template(entry, resolved))
        source_vectors[entry.label_id] = resolved
    return templates, source_vectors


templates, template_source_vectors = build_all_templates(labels, PDF_PATH)
print(f'{len(templates)} template(s) built')

## 6 - Template stats

Per-character breakdown: how many Vectors, how many total items/operations, how many flattened
segments (post curve-subdivision), and the per-item-type counts (`l`/`c`/`re`/`qu`).

In [ ]:
header = f"{'label_id':<10} {'text':<10} {'vectors':>7} {'items':>6} {'segments':>8}  item types"
print(header)
print('-' * len(header))
for t in templates:
    type_counts = ', '.join(f'{k}={v}' for k, v in t['total_item_signature'])
    print(f"{t['label_id'][:10]:<10} {t['text']:<10} {t['n_vectors']:>7} {t['n_items']:>6} "
          f"{t['n_segments']:>8}  {type_counts}")

## 7 - Save / load templates

In [ ]:
import json


def save_templates(templates, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(templates, indent=2), encoding='utf-8')


def load_templates(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


save_templates(templates, TEMPLATE_JSON_PATH)
print('saved', len(templates), 'template(s) to', TEMPLATE_JSON_PATH)

# templates = load_templates(TEMPLATE_JSON_PATH)  # uncomment to reload without rebuilding

## 8 - is_similar checks

`vector_is_similar`: op-count/type match. `cluster_is_similar`: (1) fast-reject on the combined
total signature, (2) per-Vector signature sequence must match in order (stronger than (1) alone --
needed so the flattened segment sequences actually line up before comparing deltas), (3) every
delta feature at every position must be within its configured tolerance of the template's stored
value (angle features compared mod 360 via `_normalize_angle_deg`).

In [ ]:
def vector_is_similar(v, sig):
    return _sig_key(item_type_signature(v)) == _sig_key(sig)


def cluster_is_similar(candidate_vectors, template):
    cand_total = _sig_key(_combined_signature(candidate_vectors))
    if cand_total != _sig_key(template['total_item_signature']):
        return False

    cand_per_vector = [_sig_key(item_type_signature(v)) for v in candidate_vectors]
    template_per_vector = [_sig_key(s) for s in template['per_vector_item_signatures']]
    if cand_per_vector != template_per_vector:
        return False

    segments = cluster_to_segments(candidate_vectors)
    cand_deltas = delta_chain(segments)
    if len(cand_deltas) != len(template['deltas']):
        return False

    for (len_d, ang_d, gap_ang_d, gap_dist_d), (t_len, t_ang, t_gap_ang, t_gap_dist) in zip(cand_deltas, template['deltas']):
        if abs(len_d - t_len) > LENGTH_DIFF_TOLERANCE_PCT:
            return False
        if abs(_normalize_angle_deg(ang_d - t_ang)) > ANGLE_DIFF_TOLERANCE_DEG:
            return False
        if abs(_normalize_angle_deg(gap_ang_d - t_gap_ang)) > GAP_ANGLE_TOLERANCE_DEG:
            return False
        if abs(gap_dist_d - t_gap_dist) > GAP_DIST_TOLERANCE_PCT:
            return False
    return True

## 9 - Candidate generation: spatial groups + seqno-contiguous runs

`seqno` is page-global draw order across *every* drawing on the page, not just text, so requiring
literally-consecutive integer seqnos would be far too strict. Two-stage candidate generation
instead: (1) coarse spatial grouping via the existing `cluster_spatial` (plausible text runs /
nearby drawings), then (2) within each spatial group, sort by `seqno` and split into maximal runs
wherever the *rank* gap in the page's global seqno ordering exceeds `MAX_SEQNO_RANK_GAP` (rank gap,
not literal integer gap -- tolerates a configurable number of interleaved non-group seqnos).
`MAX_SEQNO_RANK_GAP = 1` is strict adjacency; the first knob to loosen if real pages interleave
other drawings mid-character.

In [ ]:
from rastervec.commons.helpers.clustering import cluster_spatial
from rastervec.commons.helpers.geometry import union_bbox


def _seqno_rank_map(page_vectors):
    ordered = sorted(page_vectors, key=lambda v: v.seqno)
    return {id(v): rank for rank, v in enumerate(ordered)}


def seqno_contiguous_runs(group_vectors, seqno_rank, max_rank_gap=1):
    ordered = sorted(group_vectors, key=lambda v: v.seqno)
    if not ordered:
        return []
    runs = [[ordered[0]]]
    for prev, v in zip(ordered, ordered[1:]):
        if seqno_rank[id(v)] - seqno_rank[id(prev)] <= max_rank_gap:
            runs[-1].append(v)
        else:
            runs.append([v])
    return runs


def candidate_runs_for_page(page_vectors, spatial_gap, max_rank_gap):
    rank = _seqno_rank_map(page_vectors)
    groups = cluster_spatial(page_vectors, get_bbox=lambda v: v.bbox, threshold=spatial_gap)
    runs = []
    for group in groups:
        runs.extend(seqno_contiguous_runs(group, rank, max_rank_gap=max_rank_gap))
    return runs

## 10 - Sliding-window template search

For each template of `k` Vectors, slide a `k`-wide window across each candidate run (already
seqno-sorted) and test `cluster_is_similar`. Overlapping matches (sharing a `seqno`) are greedily
deduped, preferring the match spanning more Vectors.

In [ ]:
def sliding_window_matches(run, templates):
    matches = []
    for template in templates:
        k = template['n_vectors']
        if k == 0 or k > len(run):
            continue
        for start in range(0, len(run) - k + 1):
            window = run[start:start + k]
            if cluster_is_similar(window, template):
                matches.append({
                    'label_id': template['label_id'],
                    'text': template['text'],
                    'window_vectors': window,
                    'bbox': union_bbox([v.bbox for v in window]),
                })
    return matches


def _matches_overlap(a, b):
    a_seqnos = {v.seqno for v in a['window_vectors']}
    b_seqnos = {v.seqno for v in b['window_vectors']}
    return bool(a_seqnos & b_seqnos)


def dedup_matches(matches):
    ordered = sorted(matches, key=lambda m: -len(m['window_vectors']))
    kept = []
    for m in ordered:
        if not any(_matches_overlap(m, k) for k in kept):
            kept.append(m)
    return kept


def detect_page(pdf_path, page_index, templates, spatial_gap, max_rank_gap):
    with Reader(pdf_path) as reader:
        page = reader.get_page(page_index)
        page_vectors = extract_vectors(page)
    runs = candidate_runs_for_page(page_vectors, spatial_gap, max_rank_gap)
    all_matches = []
    for run in runs:
        all_matches.extend(sliding_window_matches(run, templates))
    return dedup_matches(all_matches)

## 11 - Run detection

Defaults to every page that has a vector label (a natural self-check: the labelled character
should re-detect itself, plus any other repeated occurrences on the same page).

In [ ]:
page_indices = DETECT_PAGE_INDICES
if page_indices is None:
    page_indices = sorted({e.page_index for e in vector_entries}) or [0]

all_detections = []
for page_index in page_indices:
    detections = detect_page(PDF_PATH, page_index, templates, SPATIAL_CLUSTER_GAP, MAX_SEQNO_RANK_GAP)
    print(f'page {page_index}: {len(detections)} match(es)')
    for d in detections:
        bbox = tuple(round(c, 1) for c in d['bbox'])
        print(f"  {d['text']!r} (label {d['label_id']}) bbox={bbox}")
    all_detections.extend(detections)

## 12 - Visualization: template vs. detected match

One row per detection: what the template looked like when labelled (left) vs. the detected
occurrence (right).

In [ ]:
import matplotlib.pyplot as plt

from rastervec.commons.renderer import render_vector_cluster

shown = all_detections[:MAX_MATCHES_SHOWN]
if not shown:
    print('no matches to visualize')
else:
    fig, axes = plt.subplots(len(shown), 2, figsize=(6, 3 * len(shown)))
    if len(shown) == 1:
        axes = [axes]

    for row, m in zip(axes, shown):
        ax_template, ax_match = row
        template_vecs = template_source_vectors[m['label_id']]
        ax_template.imshow(render_vector_cluster(template_vecs, RENDER_DPI))
        ax_template.set_title(f"template {m['text']!r}")
        ax_template.axis('off')

        ax_match.imshow(render_vector_cluster(m['window_vectors'], RENDER_DPI))
        bbox = tuple(round(c, 1) for c in m['bbox'])
        ax_match.set_title(f'match bbox={bbox}')
        ax_match.axis('off')

    plt.tight_layout()
    plt.show()